# Triagem Inteligente — ML + Fuzzy

Notebook **passo a passo** do trabalho. Cada etapa carrega um módulo
do pacote `triagem_fuzzy` (em `src/`) e demonstra o seu comportamento.

> ⚠️ Trabalho **estritamente acadêmico** — não constitui diagnóstico
> médico real.

**Decisões fixadas**

| ID | Decisão | Valor |
|----|---------|-------|
| D1 | Remapeamento das classes | `0→normal`, `1→atenção`, `2,3→risco` |
| D2 | Algoritmo ML | Random Forest |
| D3 | Articulação | Aproximação A (Comparação) **e** B (Integração) |
| D4 | Base | `dataset/triagem_fuzzy.csv` (18.000 linhas) |
| D5 | Seed | 50 |


## 0. Configuração — adicionar `src/` ao path

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))


## 1. Imports

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from triagem_fuzzy import config
from triagem_fuzzy.ingestion import DataIngestion
from triagem_fuzzy.preprocessing import DataPreprocessor
from triagem_fuzzy.eda import ExploratoryAnalysis
from triagem_fuzzy.ml.random_forest import TriageRandomForest
from triagem_fuzzy.ml.evaluator import ModelEvaluator
from triagem_fuzzy.fuzzy.factory import build_standalone_engine, build_integrated_engine
from triagem_fuzzy.articulation.comparator import TriageComparator
from triagem_fuzzy.articulation.integrator import TriageIntegrator


## 2. Spec 01 — Ingestão e validação

Carrega `dataset/triagem_fuzzy.csv`, valida esquema (colunas, tipos,
intervalos, categorias) e retorna o `DataFrame` bruto.


In [3]:
df = DataIngestion().load_validated()

assert df.shape == (18000, 10)
assert df.isna().sum().sum() == 0
df.head()


,age,heart_rate,systolic_blood_pressure,oxygen_saturation,body_temperature,pain_level,chronic_disease_count,previous_er_visits,arrival_mode,triage_level
0,17.9,95.4,147.1,97.4,36.48,1,0,0,walk_in,0
1,79.2,147.9,158.6,96.0,39.35,10,4,2,ambulance,3
2,51.1,87.1,128.2,98.5,37.74,5,2,2,walk_in,1
3,56.8,84.7,147.2,92.5,37.55,4,4,4,walk_in,1
4,39.2,58.0,107.8,99.0,36.26,2,1,1,walk_in,0


### 2.1 Distribuição original do alvo (4 níveis)

In [4]:
df["triage_level"].value_counts().sort_index()


triage_level
0    9924
1    4484
2    2701
3     891
Name: count, dtype: int64

## 3. Spec 02 — Pré-processamento

Aplica o remapeamento D1, one-hot de `arrival_mode`, e split
estratificado 80/20 com `random_state=50`.


In [5]:
preprocessor = DataPreprocessor()
data = preprocessor.run(df)

assert data.X_train.shape == (14400, 11)
assert data.X_test.shape == (3600, 11)
assert set(data.y_train.unique()) == {0, 1, 2}

print(f"Treino : {len(data.X_train):>5}  ({data.X_train.shape[1]} features)")
print(f"Teste  : {len(data.X_test):>5}")
print(f"Classes: {data.label_encoder}")


Treino : 14400  (11 features)
Teste  :  3600
Classes: {0: 'normal', 1: 'atencao', 2: 'risco'}


### 3.1 Balanço de classes após o remapeamento

In [6]:
counts = data.y_train.value_counts().sort_index()
counts.index = [config.TRIAGE_LABELS[i] for i in counts.index]
counts


normal     7939
atencao    3587
risco      2874
Name: count, dtype: int64

## 4. Spec 03 — Análise exploratória (EDA)

Resumo descritivo, balanço de classes e correlação entre features
numéricas.


In [7]:
eda = ExploratoryAnalysis(config.OUTPUT_DIR)
summary = eda.describe(df)
summary.round(2)


,mean,std,min,25%,50%,75%,max
age,44.72,19.10,0.00,31.20,44.00,57.50,95.00
heart_rate,83.29,16.96,33.40,71.50,81.60,93.20,152.30
systolic_blood_pressure,128.07,18.81,65.80,114.90,126.90,139.90,219.70
oxygen_saturation,96.09,3.33,79.50,94.30,96.60,98.70,100.00
body_temperature,37.22,0.91,34.47,36.58,37.12,37.77,41.13
pain_level,3.40,2.04,1.00,2.00,3.00,5.00,10.00
chronic_disease_count,1.07,1.31,0.00,0.00,1.00,2.00,10.00
previous_er_visits,1.27,1.45,0.00,0.00,1.00,2.00,11.00


### 4.1 Matriz de correlação (Pearson)

In [8]:
corr = eda.correlations(df)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, label="r")
ax.set_title("Correlação entre features numéricas")
fig.tight_layout()
plt.show()


/var/folders/xd/8_gjb3694sb8_png5kl0k5900000gp/T/ipykernel_98161/720257258.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Spec 04 — Treinamento do Random Forest

300 árvores · `class_weight='balanced'` · `min_samples_leaf=2`.


In [9]:
model = TriageRandomForest()
model.fit(data.X_train, data.y_train)

assert model.predict(data.X_test).shape == (len(data.X_test),)
proba = model.predict_proba(data.X_test)
assert np.allclose(proba.sum(axis=1), 1.0)

print(f"Modelo treinado. Classes: {model.classes_.tolist()}")


Modelo treinado. Classes: [0, 1, 2]


## 6. Spec 04 — Avaliação (métricas pedidas no enunciado)

Acurácia · Matriz de confusão · Precisão · Recall · F1-score.


In [10]:
evaluator = ModelEvaluator(data.label_encoder)
report = evaluator.evaluate(model, data.X_test, data.y_test)

print(f"Acurácia    : {report.accuracy:.4f}")
print(f"Macro F1    : {report.macro_f1:.4f}")
print(f"Weighted F1 : {report.weighted_f1:.4f}")


Acurácia    : 0.9550
Macro F1    : 0.9493
Weighted F1 : 0.9551


### 6.1 Precisão, recall e F1 por classe

In [11]:
report.per_class.round(4)


,precision,recall,f1,support
normal,0.9737,0.9688,0.9712,1985
atencao,0.9053,0.9164,0.9108,897
risco,0.9665,0.9652,0.9659,718


### 6.2 Matriz de confusão

In [12]:
cm = report.confusion_matrix
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(len(cm.columns)))
ax.set_yticks(range(len(cm.index)))
ax.set_xticklabels(cm.columns)
ax.set_yticklabels(cm.index)
ax.set_xlabel("predito")
ax.set_ylabel("verdadeiro")
thresh = cm.values.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm.values[i, j]), ha="center", va="center",
                color="white" if cm.values[i, j] > thresh else "black")
fig.colorbar(im, ax=ax)
ax.set_title("Matriz de confusão — Random Forest")
fig.tight_layout()
plt.show()
cm


/var/folders/xd/8_gjb3694sb8_png5kl0k5900000gp/T/ipykernel_98161/2294396671.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


pred,normal,atencao,risco
true,,,
normal,1923,62,0
atencao,51,822,24
risco,1,24,693


### 6.3 Importância das features (top 5)

In [13]:
report.feature_importances.head(5).round(4)


pain_level           0.4158
body_temperature     0.1510
heart_rate           0.1125
oxygen_saturation    0.0959
age                  0.0749
Name: importance, dtype: float64

## 7. Spec 05 — Sistema fuzzy de Mamdani

3 entradas vitais (`temperatura`, `frequência cardíaca`,
`pressão sistólica`) → 1 saída (`risk_score` ∈ [0, 10]). 9 regras,
agregação `max`, defuzzificação por centroide.


In [14]:
engine = build_standalone_engine()

print(f"Entradas : {sorted(engine.required_inputs())}")
print(f"Saída    : {engine.output.name}  universo={engine.output.universe}")
print(f"Regras   : {len(engine.rules)}")


Entradas : ['body_temperature', 'heart_rate', 'systolic_blood_pressure']
Saída    : risk_score  universo=(0.0, 10.0)
Regras   : 9


### 7.1 Funções de pertinência das entradas

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, var) in zip(axes, engine.inputs.items()):
    xs = var.universe_array()
    for term_name, mf in var.terms.items():
        ax.plot(xs, mf.evaluate(xs), label=term_name)
    ax.set_title(name)
    ax.set_xlabel(name)
    ax.set_ylabel("pertinência")
    ax.set_ylim(-0.05, 1.1)
    ax.legend()
fig.tight_layout()
plt.show()


/var/folders/xd/8_gjb3694sb8_png5kl0k5900000gp/T/ipykernel_98161/790947409.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.2 Função de pertinência da saída `risk_score`

In [16]:
xs = engine.output.universe_array()
fig, ax = plt.subplots(figsize=(7, 3.5))
for term_name, mf in engine.output.terms.items():
    ax.plot(xs, mf.evaluate(xs), label=term_name)
for label, (lo, hi) in config.FUZZY_DECISION_BANDS.items():
    ax.axvspan(lo, hi, alpha=0.05)
ax.set_xlabel("risk_score")
ax.set_ylabel("pertinência")
ax.set_title("Saída fuzzy e bandas de decisão")
ax.legend()
fig.tight_layout()
plt.show()


/var/folders/xd/8_gjb3694sb8_png5kl0k5900000gp/T/ipykernel_98161/4169272084.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.3 Base de regras (texto)

In [17]:
rules_df = pd.DataFrame([
    {
        "regra": r.name,
        "antecedente": " E ".join(f"{v}={t}" for v, t in r.antecedents),
        "consequente": f"{r.consequent[0]}={r.consequent[1]}",
    }
    for r in engine.rules
])
rules_df


,regra,antecedente,consequente
0,R1,body_temperature=alta E heart_rate=alta,risk_score=alto
1,R2,body_temperature=alta E systolic_blood_pressur...,risk_score=alto
2,R3,heart_rate=alta E systolic_blood_pressure=baixa,risk_score=alto
3,R4,body_temperature=normal E heart_rate=normal E ...,risk_score=baixo
4,R5,body_temperature=baixa E systolic_blood_pressu...,risk_score=alto
5,R6,heart_rate=baixa E systolic_blood_pressure=normal,risk_score=medio
6,R7,body_temperature=alta E heart_rate=normal E sy...,risk_score=medio
7,R8,heart_rate=alta E systolic_blood_pressure=normal,risk_score=medio
8,R9,body_temperature=normal E systolic_blood_press...,risk_score=medio


### 7.4 Demonstração — paciente "normal típico"

Temperatura normal, FC e PA dentro da faixa esperada → score baixo.


In [18]:
sample_normal = {"body_temperature": 36.8, "heart_rate": 75.0, "systolic_blood_pressure": 120.0}
trace_normal = engine.explain(sample_normal)

assert trace_normal.crisp_output < 3.5
assert trace_normal.classification == "normal"

print(f"risk_score     : {trace_normal.crisp_output:.3f}")
print(f"classe sugerida: {trace_normal.classification}")


risk_score     : 1.654
classe sugerida: normal


### 7.5 Demonstração — paciente "risco típico"

Febre alta, taquicardia e hipotensão → score alto.


In [19]:
sample_risco = {"body_temperature": 39.5, "heart_rate": 130.0, "systolic_blood_pressure": 85.0}
trace_risco = engine.explain(sample_risco)

assert trace_risco.crisp_output > 6.5
assert trace_risco.classification == "risco"

print(f"risk_score     : {trace_risco.crisp_output:.3f}")
print(f"classe sugerida: {trace_risco.classification}")


risk_score     : 8.444
classe sugerida: risco


### 7.6 Regras que dispararam para o paciente de risco

In [20]:
activations = pd.DataFrame([
    {
        "regra": a.rule.name,
        "antecedente": " E ".join(f"{v}={t}" for v, t in a.rule.antecedents),
        "consequente": a.rule.consequent[1],
        "ativação": round(a.strength, 3),
    }
    for a in trace_risco.activations
])
activations.sort_values("ativação", ascending=False)


,regra,antecedente,consequente,ativação
0,R1,body_temperature=alta E heart_rate=alta,alto,1.0
1,R2,body_temperature=alta E systolic_blood_pressur...,alto,1.0
2,R3,heart_rate=alta E systolic_blood_pressure=baixa,alto,1.0
3,R4,body_temperature=normal E heart_rate=normal E ...,baixo,0.0
4,R5,body_temperature=baixa E systolic_blood_pressu...,alto,0.0
5,R6,heart_rate=baixa E systolic_blood_pressure=normal,medio,0.0
6,R7,body_temperature=alta E heart_rate=normal E sy...,medio,0.0
7,R8,heart_rate=alta E systolic_blood_pressure=normal,medio,0.0
8,R9,body_temperature=normal E systolic_blood_press...,medio,0.0


## 8. Spec 06 — Articulação A (Comparação)

ML e fuzzy independentes sobre as mesmas amostras de teste.
Amostragem de 1.500 linhas para acelerar a passagem ao vivo.


In [21]:
sampled_X = data.X_test.sample(n=1500, random_state=config.RANDOM_STATE)
sampled_y = data.y_test.loc[sampled_X.index]

cmp_report = TriageComparator(model, build_standalone_engine(), data.label_encoder).run(sampled_X, sampled_y)

assert 0.0 <= cmp_report.agreement_rate <= 1.0

print(f"Concordância ML × Fuzzy : {cmp_report.agreement_rate:.3f}")
print(f"Acurácia ML            : {cmp_report.ml_accuracy:.3f}")
print(f"Acurácia Fuzzy         : {cmp_report.fuzzy_accuracy:.3f}")


Concordância ML × Fuzzy : 0.578
Acurácia ML            : 0.955
Acurácia Fuzzy         : 0.571


### 8.1 Matriz ML × Fuzzy

In [22]:
cmp_report.confusion_ml_vs_fuzzy


fuzzy,normal,atencao,risco
ml,,,
normal,426,397,21
atencao,86,250,42
risco,2,85,191


### 8.2 Concordância por classe verdadeira

In [23]:
cmp_report.per_class_agreement.round(3)


y_true
normal     0.509
atencao    0.652
risco      0.689
Name: agree, dtype: float64

## 9. Spec 06 — Articulação B (Integração)

A probabilidade `P(risco)` do Random Forest vira a 4ª entrada
linguística do motor fuzzy integrado.


In [24]:
int_report = TriageIntegrator(model, build_integrated_engine(), data.label_encoder).run(sampled_X, sampled_y)

print(f"Acurácia ML       : {int_report.accuracy_ml:.3f}")
print(f"Acurácia integrada: {int_report.accuracy_integrated:.3f}")
print(f"Macro F1 ML       : {int_report.macro_f1_ml:.3f}")
print(f"Macro F1 integrado: {int_report.macro_f1_integrated:.3f}")


Acurácia ML       : 0.955
Acurácia integrada: 0.691
Macro F1 ML       : 0.950
Macro F1 integrado: 0.683


### 9.1 Δ por classe (recall e F1 — integrado menos ML)

In [25]:
int_report.delta_per_class.round(3)


,recall_ml,recall_integrated,delta_recall,f1_ml,f1_integrated,delta_f1
normal,0.965,0.715,-0.250,0.969,0.744,-0.226
atencao,0.926,0.422,-0.504,0.910,0.409,-0.501
risco,0.961,0.965,0.004,0.970,0.897,-0.073


### 9.2 Matriz de confusão do sistema integrado

In [26]:
int_report.confusion_integrated


pred,normal,atencao,risco
true,,,
normal,609,227,16
atencao,174,154,37
risco,3,7,273


### 9.3 Distribuição do `risk_score` integrado

In [27]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(int_report.predictions["integrated_score"], bins=40, edgecolor="black")
ax.set_xlabel("risk_score (integrado)")
ax.set_ylabel("frequência")
ax.set_title("Distribuição do risk_score integrado")
fig.tight_layout()
plt.show()


/var/folders/xd/8_gjb3694sb8_png5kl0k5900000gp/T/ipykernel_98161/2385327167.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Resumo final

Reúne em uma única tabela o que será apresentado ao professor.


In [28]:
resumo = pd.DataFrame(
    {
        "ML puro":   [report.accuracy, report.macro_f1],
        "Fuzzy só":  [cmp_report.fuzzy_accuracy, np.nan],
        "Integrado": [int_report.accuracy_integrated, int_report.macro_f1_integrated],
    },
    index=["acurácia", "macro F1"],
)
resumo.round(3)


,ML puro,Fuzzy só,Integrado
acurácia,0.955,0.571,0.691
macro F1,0.949,NaN,0.683


## Conclusão

- **ML puro** entrega a maior acurácia (≈0,955), com erros concentrados
  nas fronteiras adjacentes entre classes.
- **Fuzzy independente** é mais limitado (≈0,57) porque só usa três
  variáveis vitais, mas suas decisões são totalmente interpretáveis
  pelas 9 regras `SE … ENTÃO`.
- **Integração (B)** reduz a acurácia agregada mas **preserva o recall
  da classe `risco`** — escolha clínica defensável: errar para mais
  é menos custoso que perder pacientes graves.

> Demonstração completa, interativa, com simulador de pacientes:
> `uv run streamlit run app.py`.
